In [3]:
import sys
import os
import json
import pandas as pd
import numpy as np

# Ensure root directory is accessible for custom package imports
root_dir = os.path.abspath("../..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from machine_learning.preprocessing import (
    load_and_clean_data,
    resample_to_hourly_grid,
    train_test_chronological_split
)
from machine_learning.feature_engineering import build_feature_pipeline
from machine_learning.model_utils import create_supervised_matrices
from machine_learning.models import (
    get_model_pipeline,
    tune_time_series_hyperparameters,
    save_trained_model
)
from machine_learning.train import calculate_metrics

In [4]:
# 1. Load & clean raw dataset
raw_df = load_and_clean_data(file_path="../../data/raw_sample_data.csv")

# 2. Resample to continuous hourly grid
hourly_df = resample_to_hourly_grid(raw_df)

# 3. Generate feature matrix (cyclical time, lags, shifted rolling stats)
target_pollutants = ['pm10', 'pm25']
feature_df = build_feature_pipeline(
    df=hourly_df,
    target_cols=target_pollutants,
    lags=[1, 2, 3, 6, 12, 24],
    windows=[3, 6, 12, 24],
    drop_na=True
)

# Flatten MultiIndex column names if necessary to ensure clean X feature matrix headers
if isinstance(feature_df.columns, pd.MultiIndex):
    feature_df.columns = ['_'.join(col).strip('_') for col in feature_df.columns.values]

print(f"Engineered Feature Matrix Shape: {feature_df.shape}")
feature_df.head()

Engineered Feature Matrix Shape: (100, 17)


,bengaluru_pm10,bengaluru_pm25,chennai_pm10,chennai_pm25,delhi_pm10,delhi_pm25,mumbai_pm10,mumbai_pm25,hour,dayofweek,month,sin_hour,cos_hour,sin_dayofweek,cos_dayofweek,sin_month,cos_month
timestamp,,,,,,,,,,,,,,,,,
2026-10-01 00:00:00+00:00,127.580582,95.537338,103.600171,59.667981,93.812188,143.100003,35.423680,45.251868,0,3,10,0.000000,1.000000,0.433884,-0.900969,-0.866025,0.5
2026-10-01 01:00:00+00:00,103.456986,125.166005,137.860173,98.005722,82.142245,109.130161,80.824415,31.469196,1,3,10,0.258819,0.965926,0.433884,-0.900969,-0.866025,0.5
2026-10-01 02:00:00+00:00,40.343572,113.162324,84.755446,136.343463,70.472303,52.593914,109.013409,107.531115,2,3,10,0.500000,0.866025,0.433884,-0.900969,-0.866025,0.5
2026-10-01 03:00:00+00:00,93.226687,42.472593,16.123470,78.040069,37.954330,73.849798,73.117512,140.811382,3,3,10,0.707107,0.707107,0.433884,-0.900969,-0.866025,0.5
2026-10-01 04:00:00+00:00,106.941061,69.947946,106.948996,101.320831,142.843975,33.873377,92.778561,131.785491,4,3,10,0.866025,0.500000,0.433884,-0.900969,-0.866025,0.5


In [5]:
horizons = [1, 24]
model_types = ["xgboost", "lightgbm"]
results_summary = {}

for target in target_pollutants:
    results_summary[target] = {}
    
    # Identify target column name after column flattening
    target_col = [c for c in feature_df.columns if target in c and 'lag' not in c and 'roll' not in c and 'sin' not in c and 'cos' not in c][0]
    
    for h in horizons:
        results_summary[target][f"horizon_{h}h"] = {}
        
        # 1. Generate (X, y) pair for forecasting horizon h
        X, y = create_supervised_matrices(feature_df, target_col=target_col, horizon=h)
        
        # 2. Strict chronological train/test split (80% train / 20% test)
        train_size = int(len(X) * 0.8)
        X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
        y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
        
        for m_type in model_types:
            print(f"\n⚡ Training {m_type.upper()} for {target.upper()} | {h}h Horizon...")
            
            # Tune hyperparameters with TimeSeriesSplit CV
            best_model, best_params = tune_time_series_hyperparameters(
                X_train, y_train, model_type=m_type, n_splits=3
            )
            
            # Predict on holdout test set
            y_pred = best_model.predict(X_test)
            metrics = calculate_metrics(y_test.values, y_pred)
            
            results_summary[target][f"horizon_{h}h"][m_type] = metrics
            print(f"📈 [{m_type.upper()} Performance] MAE: {metrics['MAE']} | RMSE: {metrics['RMSE']} | MAPE: {metrics['MAPE_percent']}%")
            
            # Save trained binary artifact
            save_trained_model(best_model, model_name=f"{m_type}_{target}_h{h}h")


⚡ Training XGBOOST for PM10 | 1h Horizon...
✅ Best XGBOOST Parameters: {'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 100}
📈 [XGBOOST Performance] MAE: 36.417 | RMSE: 41.9787 | MAPE: 75.7638%
📦 Saved model checkpoint to: machine_learning/artifacts/models/xgboost_pm10_h1h.joblib

⚡ Training LIGHTGBM for PM10 | 1h Horizon...
✅ Best LIGHTGBM Parameters: {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 50, 'num_leaves': 15}
📈 [LIGHTGBM Performance] MAE: 36.7803 | RMSE: 42.4139 | MAPE: 85.4129%
📦 Saved model checkpoint to: machine_learning/artifacts/models/lightgbm_pm10_h1h.joblib

⚡ Training XGBOOST for PM10 | 24h Horizon...
✅ Best XGBOOST Parameters: {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 50}
📈 [XGBOOST Performance] MAE: 33.8787 | RMSE: 41.2299 | MAPE: 91.9051%
📦 Saved model checkpoint to: machine_learning/artifacts/models/xgboost_pm10_h24h.joblib

⚡ Training LIGHTGBM for PM10 | 24h Horizon...
✅ Best LIGHTGBM Parameters: {'learning_rate': 0.1, 'max_depth'

In [11]:
import json
import os
import pandas as pd
import numpy as np

# 1. Compute Naive Persistence metrics directly on test set
baseline_metrics = {}
horizons = [1, 24]

for target in target_pollutants:
    baseline_metrics[target] = {}
    # Identify main target column in X matrix
    target_cols = [c for c in feature_df.columns if target in c and 'lag' not in c and 'roll' not in c and 'sin' not in c and 'cos' not in c]
    target_col = target_cols[0]
    
    for h in horizons:
        X, y = create_supervised_matrices(feature_df, target_col=target_col, horizon=h)
        train_size = int(len(X) * 0.8)
        
        y_test = y.iloc[train_size:]
        # Naive Persistence prediction: y_hat(t+h) = y(t) from feature matrix X
        y_pred_naive = X.iloc[train_size:][target_col]
        
        metrics = calculate_metrics(y_test.values, y_pred_naive.values)
        baseline_metrics[target][f"horizon_{h}h"] = metrics

# Save populated baseline metrics JSON artifact
os.makedirs("../artifacts", exist_ok=True)
with open("../artifacts/baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=4)

print("✅ Baseline metrics computed and saved to ../artifacts/baseline_metrics.json")
print("=" * 75)
print("🏆 MODEL PERFORMANCE VS NAIVE PERSISTENCE BASELINE")
print("=" * 75)

# 2. Build comparison table
comparison_rows = []

for target in target_pollutants:
    for h in horizons:
        h_key = f"horizon_{h}h"
        base_mape = baseline_metrics[target][h_key]["MAPE_percent"]
        
        for m_type in ["xgboost", "lightgbm"]:
            if target in results_summary and h_key in results_summary[target] and m_type in results_summary[target][h_key]:
                model_mape = results_summary[target][h_key][m_type]["MAPE_percent"]
                improvement = base_mape - model_mape
                
                comparison_rows.append({
                    "Target": target.upper(),
                    "Horizon": f"{h}h",
                    "Model": m_type.upper(),
                    "Baseline MAPE (%)": base_mape,
                    "Model MAPE (%)": model_mape,
                    "Absolute Improvement (%)": round(improvement, 2)
                })

df_comparison = pd.DataFrame(comparison_rows)
display(df_comparison)

✅ Baseline metrics computed and saved to ../artifacts/baseline_metrics.json
🏆 MODEL PERFORMANCE VS NAIVE PERSISTENCE BASELINE


,Target,Horizon,Model,Baseline MAPE (%),Model MAPE (%),Absolute Improvement (%)
0,PM10,1h,XGBOOST,101.9480,75.7638,26.18
1,PM10,1h,LIGHTGBM,101.9480,85.4129,16.54
2,PM10,24h,XGBOOST,101.2109,91.9051,9.31
3,PM10,24h,LIGHTGBM,101.2109,101.4118,-0.20
4,PM25,1h,XGBOOST,113.0512,112.9079,0.14
5,PM25,1h,LIGHTGBM,113.0512,104.8637,8.19
6,PM25,24h,XGBOOST,157.1831,125.9139,31.27
7,PM25,24h,LIGHTGBM,157.1831,122.9691,34.21
